# TA1: Predicting Stock Prices Using RNN with LSTM

**Objective:** Develop a Recurrent Neural Network (RNN) with Long Short-Term Memory (LSTM) units in Python to predict future stock prices based on historical data.

**Dataset:** S&P 500 subset — Apple Inc. (AAPL) daily stock data from **2019-01-01 to 2024-01-01** (5 years), downloaded via `yfinance`.

**Sections:**
1. Data Preparation
2. Model Development
3. Training
4. Prediction
5. Evaluation (MAE, RMSE, MAPE) & Visualizations (line + candlestick)

## 0. Install & Import Libraries

In [ ]:
# Uncomment the next line if any library is missing
# !pip install yfinance tensorflow scikit-learn pandas numpy matplotlib plotly

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

import plotly.graph_objects as go

np.random.seed(42)
tf.random.set_seed(42)
print('TensorFlow version:', tf.__version__)

## 1. Data Preparation
Choose AAPL (member of S&P 500). Download 5 years of historical data, perform feature selection, normalization, and create sequences for the RNN model.

In [ ]:
# 1a. Download historical stock data
TICKER = 'AAPL'
START = '2019-01-01'
END   = '2024-01-01'

df = yf.download(TICKER, start=START, end=END, progress=False)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)
df = df.dropna()
print('Shape:', df.shape)
df.head()

In [ ]:
# 1b. Quick look at the Close price
plt.figure(figsize=(12,4))
plt.plot(df.index, df['Close'], color='steelblue')
plt.title(f'{TICKER} Close Price (2019-2024)')
plt.xlabel('Date'); plt.ylabel('Close Price (USD)')
plt.grid(alpha=0.3); plt.show()

In [ ]:
# 1c. Feature engineering — work with LOG RETURNS (stationary target)
#     This avoids the out-of-range problem when test prices exceed train max,
#     and prevents the iterative forecast from collapsing toward the train-mean.
prices  = df['Close'].astype('float32').values            # raw close prices
log_ret = np.log(prices[1:] / prices[:-1]).astype('float32')   # length N-1
ret_dates = df.index[1:]
print('Returns shape:', log_ret.shape, ' mean:', log_ret.mean(), ' std:', log_ret.std())

# 1d. Train/Test split (80/20, chronological) on returns
split_idx = int(len(log_ret) * 0.80)
train_ret = log_ret[:split_idx].reshape(-1, 1)
test_ret  = log_ret[split_idx:].reshape(-1, 1)
print(f'Train returns: {len(train_ret)}, Test returns: {len(test_ret)}')

# 1e. Standardize returns (fit scaler on TRAIN only — no leakage)
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_ret)
test_scaled  = scaler.transform(test_ret)

In [ ]:
# 1f. Create sliding-window sequences over RETURNS
WINDOW = 60  # use last 60 days of returns to predict the next-day return

def make_sequences(series, window):
    X, y = [], []
    for i in range(window, len(series)):
        X.append(series[i-window:i, 0])
        y.append(series[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = make_sequences(train_scaled, WINDOW)

# For test, prepend the tail of train so the first test window is complete
full_scaled = np.concatenate([train_scaled, test_scaled], axis=0)
test_input  = full_scaled[len(train_scaled) - WINDOW:]
X_test, y_test = make_sequences(test_input, WINDOW)

# Reshape for LSTM: (samples, timesteps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test  = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
print('X_train:', X_train.shape, '  y_train:', y_train.shape)
print('X_test :', X_test.shape,  '  y_test :', y_test.shape)

## 2. Model Development
Build a stacked LSTM (two LSTM layers) with Dropout regularization using TensorFlow / Keras.

In [ ]:
def build_lstm(window):
    model = Sequential([
        Input(shape=(window, 1)),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
    return model

model = build_lstm(WINDOW)
model.summary()

## 3. Training
Train with Adam optimizer, MSE loss, and EarlyStopping to prevent overfitting.

In [ ]:
es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[es],
    verbose=1
)

In [ ]:
# Training curves
plt.figure(figsize=(10,4))
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.legend(); plt.grid(alpha=0.3); plt.show()

## 4. Prediction
Predict on the test set and also forecast future prices starting from the last known window.

In [ ]:
# 4a. One-step-ahead predictions on the test set
#     Model outputs scaled log-returns -> inverse-scale -> reconstruct prices
#     using the PREVIOUS ACTUAL price (true one-step-ahead evaluation).
pred_scaled_ret = model.predict(X_test, verbose=0)
pred_log_ret    = scaler.inverse_transform(pred_scaled_ret).flatten()
true_log_ret    = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

# Price at day t-1 for each predicted return at day t.
# Returns index split_idx corresponds to price index split_idx+1 (since log_ret = log(P[1:]/P[:-1])).
# So previous prices are df['Close'].iloc[split_idx : split_idx + len(pred_log_ret)].
prev_prices = df['Close'].iloc[split_idx : split_idx + len(pred_log_ret)].values
predictions = (prev_prices * np.exp(pred_log_ret)).reshape(-1, 1)
actuals     = (prev_prices * np.exp(true_log_ret)).reshape(-1, 1)  # == df['Close'] one step later

test_dates = df.index[split_idx + 1 : split_idx + 1 + len(predictions)]

results = pd.DataFrame({
    'Date'     : test_dates,
    'Actual'   : actuals.flatten(),
    'Predicted': predictions.flatten()
}).set_index('Date')
results.head()

In [ ]:
# 4b. Multi-step future forecast (short horizon — iterative rollouts of returns)
#     Iterative forecasts compound errors, so keep horizon small (≤ 2 weeks).
FUTURE_DAYS = 10

last_window = full_scaled[-WINDOW:].copy()  # last WINDOW scaled returns
future_scaled_ret = []

cur = last_window.copy()
for _ in range(FUTURE_DAYS):
    x = cur.reshape(1, WINDOW, 1)
    nxt = model.predict(x, verbose=0)[0, 0]
    future_scaled_ret.append(nxt)
    cur = np.append(cur[1:], [[nxt]], axis=0)

future_log_ret = scaler.inverse_transform(np.array(future_scaled_ret).reshape(-1, 1)).flatten()

# Reconstruct prices by compounding returns from the last known close
last_price    = float(df['Close'].iloc[-1])
future_prices = last_price * np.exp(np.cumsum(future_log_ret))

future_dates = pd.bdate_range(start=df.index[-1] + pd.Timedelta(days=1), periods=FUTURE_DAYS)
future_df = pd.DataFrame({'Forecast': future_prices}, index=future_dates)
print('Last known close:', round(last_price, 2))
future_df

## 5. Evaluation & Visualizations

In [ ]:
# 5a. Metrics: MAE, RMSE, MAPE
mae  = mean_absolute_error(actuals, predictions)
rmse = np.sqrt(mean_squared_error(actuals, predictions))
mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100

print(f'MAE : {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'MAPE: {mape:.4f}%')

In [ ]:
# 5b. Line chart: Actual vs Predicted (test) + Future forecast
plt.figure(figsize=(14, 6))
plt.plot(df.index[:split_idx + 1], df['Close'].iloc[:split_idx + 1], label='Train (Actual)', color='gray', alpha=0.6)
plt.plot(results.index, results['Actual'],    label='Test Actual',    color='steelblue')
plt.plot(results.index, results['Predicted'], label='Test Predicted (1-step)', color='orange')
plt.plot(future_df.index, future_df['Forecast'], label=f'Next {FUTURE_DAYS} Days Forecast', color='red', linestyle='--', marker='o')
plt.title(f'{TICKER} — LSTM Stock Price Prediction (returns-based)')
plt.xlabel('Date'); plt.ylabel('Close Price (USD)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# 5c. Zoomed view: test period only
plt.figure(figsize=(14, 5))
plt.plot(results.index, results['Actual'],    label='Actual',    color='steelblue', linewidth=2)
plt.plot(results.index, results['Predicted'], label='Predicted', color='orange',    linewidth=2)
plt.title(f'{TICKER} — Actual vs Predicted (Test Set)')
plt.xlabel('Date'); plt.ylabel('Close Price (USD)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# 5d. Candlestick chart (last 6 months of actuals + future forecast overlay)
recent = df.iloc[-126:]  # ~6 months of trading days

fig = go.Figure(data=[go.Candlestick(
    x=recent.index,
    open =recent['Open'],
    high =recent['High'],
    low  =recent['Low'],
    close=recent['Close'],
    name='Historical OHLC'
)])

fig.add_trace(go.Scatter(
    x=future_df.index, y=future_df['Forecast'],
    mode='lines+markers', name=f'LSTM Forecast ({FUTURE_DAYS}d)',
    line=dict(color='red', width=2, dash='dash')
))

fig.update_layout(
    title=f'{TICKER} Candlestick Chart + LSTM Forecast (returns-based)',
    xaxis_title='Date', yaxis_title='Price (USD)',
    xaxis_rangeslider_visible=False, height=600
)
fig.show()

## Observations & Notes

- **Why returns instead of raw prices?** Stock prices are non-stationary and trend upward over years. An LSTM trained on raw prices fails when test prices exceed the training max (out-of-range inputs after MinMax scaling), and iterative multi-step rollouts collapse toward the training mean — producing a fake "crash". Log-returns are approximately stationary, so the model generalizes to any price level and forecasts follow recent volatility instead of drifting downward.
- **Reconstruction:** `P_t = P_{t-1} * exp(r_t)`. For the test set we use the **actual** `P_{t-1}` (true one-step-ahead). For the future forecast we compound predicted returns from the last known close.
- **Model:** Stacked LSTM (64 → 64) with Dropout(0.2), Dense(32, ReLU) head, Adam + MSE on standardized log-returns, EarlyStopping patience=8.
- **Input window:** 60 days of returns → next-day return.
- **Forecast horizon:** capped at **10 business days** — iterative LSTM rollouts compound errors and become unreliable beyond ~2 weeks.
- **Metrics:** MAE, RMSE, MAPE on reconstructed prices.
- **Hyperparameters to try:** window size (30/60/90), hidden units (32/64/128), dropout rate, batch size, learning rate, and multivariate features (Volume, OHLC, RSI, MACD, rolling means).